# Cross-Lingual Semantic Search Demo

This notebook demonstrates multilingual semantic retrieval using transformer embeddings and vector search.

Research Question:

Can multilingual embeddings retrieve semantically equivalent documents across languages without translation?

Pipeline:
```
Query 
↓
Multilingual Encoder
↓
Semantic Vector
↓
FAISS Similarity Search
↓
Cross-Lingual Results
```
This experiment evaluates the retrieval layer before advanced ranking and linguistic error analysis.

### Import Libraries and Project Components

In [2]:
import sys
from pathlib import Path

parent_dir = str(Path.cwd().parent)
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from embeddings.encoder import MultilingualEncoder
from indexing.document_store import DocumentStore
from indexing.vector_index import VectorIndex
from retrieval.semantic_search import SemanticSearchEngine

import pandas as pd

### Initialize Encoder

In [3]:
# Load multilingual embedding encoder
# Encoder maps English, German, and Russian sentences to a shared embedding space

encoder = MultilingualEncoder()

print("Embedding model loaded.")

Loading embedding model: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Embedding model loaded.


### Create Multilingual Document Collection

In [14]:
# Create a small multilingual document collection.
# Each document represents the same semantic concept in different languages.

documents = [
    {
        "language": "English",
        "text": "The government introduced a new environmental policy.",
        "metadata": {
            "concept": "environment_policy"
        }
    },
    {
        "language": "German",
        "text": "Die Regierung führte eine neue Umweltpolitik ein.",
        "metadata": {
            "concept": "environment_policy"
        }
    },
    {
        "language": "Russian",
        "text": "Правительство ввело новую экологическую политику.",
        "metadata": {
            "concept": "environment_policy"
        }
    },
    {
        "language": "German",
        "text": "Das Unternehmen entwickelt erneuerbare Energietechnologien.",
        "metadata": {
            "concept": "renewable_energy"
        }
    },
    {
        "language": "Russian",
        "text": "Компания разрабатывает технологии возобновляемой энергии.",
        "metadata": {
            "concept": "renewable_energy"
        }
    }
]

# Create DataFrame to hold documents for easier visualization and manipulation
documents_df = pd.DataFrame(
    [
        {
            "language": doc["language"],
            "text": doc["text"],
            "concept": doc["metadata"]["concept"]
        }
        for doc in documents
    ]
)

documents_df

,language,text,concept
0,English,The government introduced a new environmental ...,environment_policy
1,German,Die Regierung führte eine neue Umweltpolitik ein.,environment_policy
2,Russian,Правительство ввело новую экологическую политику.,environment_policy
3,German,Das Unternehmen entwickelt erneuerbare Energie...,renewable_energy
4,Russian,Компания разрабатывает технологии возобновляем...,renewable_energy


### Build Document Store

In [15]:
# Store documents and metadata.
# Separates raw text from vector retrieval.

document_store = DocumentStore()
# Iterate through documents and add them to document store
for document in documents:
    document_store.add_document(
        text = document["text"],
        language = document["language"],
        metadata = document["metadata"]
    )

print(f"Stored documents: {len(document_store)}")

Stored documents: 5


### Generate Document Embeddings

In [6]:
# Generate all document into multilingual embeddings.
# Documents with equivalent meanings should produce nearby vectors.

document_texts = [document["text"] for document in documents]
document_embeddings = encoder.encode(document_texts)

print(f"Embedding Shape: {document_embeddings.shape}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding Shape: (5, 768)


### Create FAISS Vector Index

In [7]:
# Create vector index.
# FAISS performs nearest-neighbor similarity search (approximates cosine similarity)

embedding_dimension = document_embeddings.shape[1]
vector_index = VectorIndex(embedding_dimension)

vector_index.add(document_embeddings)

print(f"Indexed Vectors: {len(vector_index)}")

Indexed Vectors: 5


### Initialize Semantic Search Engine

In [8]:
# Combine multilingual encoder, FAISS vector index, and document metadata store
# into the complete retrieval pipeline.

search_engine = SemanticSearchEngine(
    encoder = encoder,
    vector_index = vector_index,
    doc_store = document_store
)

print("Semantic Search Engine Ready.")

Semantic Search Engine Ready.


### Cross-Lingual Retrieval Experiment
## Query 1: English → German/Russian Retrieval

The query is written in English.

The system should retrieve semantically equivalent documents even though the \
stored documents are in different languages.

In [16]:
query = "The government created a new environmental policy."

# Search for top 5 most similar documents to query in multilingual document collection.
results = search_engine.search(query, top_k = 5)
retrieval_results = [] # Initialize list to hold retrieval results

# Iterate through search results and extract relevant information for each result
for result in results:
    retrieval_results.append(
        {
            "score": round(float(result.score), 4),
            "language": result.document.language,
            "text": result.document.text,
            "concept": result.document.metadata["concept"]
        }
    )

# Create DataFrame to hold retrieval results for easier visualization and manipulation
results_df = pd.DataFrame(retrieval_results)
results_df

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,score,language,text,concept
0,0.9858,German,Die Regierung führte eine neue Umweltpolitik ein.,environment_policy
1,0.9836,English,The government introduced a new environmental ...,environment_policy
2,0.9682,Russian,Правительство ввело новую экологическую политику.,environment_policy
3,0.3552,Russian,Компания разрабатывает технологии возобновляем...,renewable_energy
4,0.3390,German,Das Unternehmen entwickelt erneuerbare Energie...,renewable_energy


### Query 2: German → Russian Retrieval
The second experiment reverses the direction.

A German query should retrieve Russian and English documents with equivalent meaning.

In [17]:
query = "Die Regierung führte eine neue Umweltpolitik ein."

# Search for top 5 most similar documents to query in multilingual document collection.
results = search_engine.search(query, top_k = 5)
retrieval_results = [] # Initialize list to hold retrieval results

# Iterate through search results and extract relevant information for each result
for result in results:
    retrieval_results.append(
        {
            "score": round(float(result.score), 4),
            "language": result.document.language,
            "text": result.document.text,
            "concept": result.document.metadata["concept"]
        }
    )

# Create DataFrame to hold retrieval results for easier visualization and manipulation
pd.DataFrame(retrieval_results)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,score,language,text,concept
0,1.0000,German,Die Regierung führte eine neue Umweltpolitik ein.,environment_policy
1,0.9910,English,The government introduced a new environmental ...,environment_policy
2,0.9862,Russian,Правительство ввело новую экологическую политику.,environment_policy
3,0.3689,Russian,Компания разрабатывает технологии возобновляем...,renewable_energy
4,0.3506,German,Das Unternehmen entwickelt erneuerbare Energie...,renewable_energy


### Query 3: Semantic Rather Than Lexical Matching

This experiment tests whether the system retrieves related concepts even when exact words differ.

Example:

Query:
"clean energy technology"

Possible retrieval:

German:
"erneuerbare Energietechnologien"

Russian:
"технологии возобновляемой энергии"

No direct lexical overlap exists.

In [18]:
query = "clean energy technology"

# Search for top 5 most similar documents to query in multilingual document collection.
results = search_engine.search(query, top_k = 5)
semantic_results = [] # Initialize list to hold retrieval results

# Iterate through search results and extract relevant information for each result
for result in results:
    semantic_results.append(
        {
            "score": round(float(result.score),4),
            "language": result.document.language,
            "text": result.document.text,
            "concept": result.document.metadata["concept"]
        }
    )

# Create DataFrame to hold retrieval results for easier visualization and manipulation
pd.DataFrame(semantic_results)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,score,language,text,concept
0,0.8307,German,Das Unternehmen entwickelt erneuerbare Energie...,renewable_energy
1,0.8195,Russian,Компания разрабатывает технологии возобновляем...,renewable_energy
2,0.4257,Russian,Правительство ввело новую экологическую политику.,environment_policy
3,0.3859,German,Die Regierung führte eine neue Umweltpolitik ein.,environment_policy
4,0.3734,English,The government introduced a new environmental ...,environment_policy


### Observations

This demo evaluates the first research question:

> Can multilingual embeddings align meaning across languages?

Expected observations:

1. Translation-equivalent sentences cluster together.
2. Retrieval is based on semantic similarity rather than exact words.
3. Different writing systems (Latin/Cyrillic) can occupy the same semantic space.
4. Remaining errors motivate later experiments:
   - linguistic error analysis
   - morphology effects
   - ranking improvements

### Add Semantic Ranking Layer

In [19]:
from retrieval.ranking import SemanticRanker

# Retrieve initial candidates using FAISS
query = "The government created a new environmental policy."

# Search for top 5 most similar documents to query in multilingual document collection.
retrieved_results = search_engine.search(query, top_k = 5)

# Apply reranking
ranker = SemanticRanker()
ranked_results = ranker.rank(retrieved_results)
ranking_output = [] # Initialize list to hold ranking results

# Iterate through ranked results and extract relevant information for each result
for result in ranked_results:
    ranking_output.append(
        {
            "retrieval_score": round(float(result.retrieval_score), 4),
            "final_score": round(float(result.final_score), 4),
            "language": result.document.language,
            "text": result.document.text,
            "reason": result.ranking_reason
        }
    )

# Create DataFrame to hold ranking results for easier visualization and manipulation
pd.DataFrame(ranking_output)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,retrieval_score,final_score,language,text,reason
0,0.9858,0.9858,German,Die Regierung führte eine neue Umweltpolitik ein.,semantic_similarity
1,0.9836,0.9836,English,The government introduced a new environmental ...,semantic_similarity
2,0.9682,0.9682,Russian,Правительство ввело новую экологическую политику.,semantic_similarity
3,0.3552,0.3552,Russian,Компания разрабатывает технологии возобновляем...,semantic_similarity
4,0.3390,0.3390,German,Das Unternehmen entwickelt erneuerbare Energie...,semantic_similarity


### Linguistic Analysis Preview

The retrieval system separates retrieval from interpretation.

FAISS determines:
- which documents are semantically closest

The linguistic analysis layer determines:
- why a retrieval succeeded or failed
- what language-specific factors influenced similarity

This demonstrates the transition from:
    
semantic retrieval

to:

computational linguistic analysis.

In [20]:
from analysis.linguistic_analysis import LinguisticAnalyzer

# Analyze the top retrieved result against the expected text for linguistic errors.
analyzer = LinguisticAnalyzer()

# Example:
# Query expects the German environmental policy document
query = "The government created a new environmental policy."
expected_text = (
    "Die Regierung führte eine neue Umweltpolitik ein."
)

# Use top retrieved result for analysis
top_result = results[0]

# Analyze the top retrieved result against the expected text for linguistic errors.
errors = analyzer.analyze(
    query = query,
    retrieved_result = top_result,
    expected_text = expected_text
)

analysis_output = [] # Initialize list to hold analysis results

# Iterate through errors and extract relevant information for each error
for error in errors:
    analysis_output.append(
        {
            "category": error.category.value,
            "explanation": error.explanation,
            "query": error.query,
            "retrieved": error.retrieved_text,
            "expected": error.expected_text
        }
    )

# Create DataFrame to hold analysis results for easier visualization and manipulation
pd.DataFrame(analysis_output)

,category,explanation,query,retrieved,expected
0,unknown,Retrieval differs from expectation but no ling...,The government created a new environmental pol...,Das Unternehmen entwickelt erneuerbare Energie...,Die Regierung führte eine neue Umweltpolitik ein.
